# CPU Usage Dashboard (CPU Bitset)

This notebook runs `pandas_cpu_bitset.py` and samples external process/system CPU usage over time.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil


def _safe_cwd():
    try:
        return Path.cwd()
    except FileNotFoundError:
        return None


candidates = []
env_script = os.environ.get('SCRIPT_PATH')
if env_script:
    candidates.append(Path(env_script))

cwd = _safe_cwd()
if cwd is not None:
    candidates.append(cwd / 'pandas_cpu_bitset.py')
    candidates.append(cwd / 'notebooks' / 'cpu_cores' / 'pandas_cpu_bitset.py')

script_path = next((p.resolve() for p in candidates if p.exists()), None)
if script_path is None:
    raise FileNotFoundError('Could not find pandas_cpu_bitset.py. Set SCRIPT_PATH env var to its absolute path.')

sample_interval_s = 0.25
active_core_threshold_pct = 5.0
repeat_factor = 20

print(f'Using script: {script_path}')
print(
    f'sample_interval_s={sample_interval_s}, '
    f'active_core_threshold_pct={active_core_threshold_pct}, '
    f'repeat_factor={repeat_factor}'
)


In [ ]:
cmd = [
    sys.executable,
    str(script_path),
    '--repeat-factor', str(repeat_factor),
]

proc = subprocess.Popen(
    cmd,
    cwd=str(script_path.parent),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

p = psutil.Process(proc.pid)
_ = p.cpu_percent(interval=None)  # warm-up

rows = []
per_core_samples = []
thread_prev = {}
started = time.perf_counter()

while proc.poll() is None:
    system_per_core = psutil.cpu_percent(interval=sample_interval_s, percpu=True)
    now = time.perf_counter()
    elapsed = now - started

    try:
        proc_cpu = p.cpu_percent(interval=None)
        thread_times = {t.id: (t.user_time + t.system_time) for t in p.threads()}
        active_threads = 0
        for tid, ttime in thread_times.items():
            prev = thread_prev.get(tid)
            if prev is not None and (ttime - prev) > 0:
                active_threads += 1
        thread_prev = thread_times
        num_threads = p.num_threads()
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        break

    system_active_cores = sum(v >= active_core_threshold_pct for v in system_per_core)
    per_core_samples.append(system_per_core)
    rows.append({
        'time_s': elapsed,
        'process_cpu_percent': proc_cpu,
        'process_cores_estimate': proc_cpu / 100.0,
        'system_active_cores': system_active_cores,
        'system_mean_cpu_percent': float(np.mean(system_per_core)),
        'num_threads': num_threads,
        'active_threads_estimate': active_threads,
    })

stdout_text = proc.communicate()[0] if proc.stdout is not None else ''
return_code = proc.returncode
df = pd.DataFrame(rows)

print(f'Process return code: {return_code}')
print(f'Samples collected: {len(df)}')
if not df.empty:
    display(df.head())


In [ ]:
if df.empty:
    raise RuntimeError('No samples were collected. Try increasing repeat_factor.')

summary = {
    'duration_s': float(df['time_s'].max()),
    'max_system_active_cores': int(df['system_active_cores'].max()),
    'mean_system_active_cores': float(df['system_active_cores'].mean()),
    'max_process_cpu_percent': float(df['process_cpu_percent'].max()),
    'mean_process_cpu_percent': float(df['process_cpu_percent'].mean()),
    'max_process_cores_estimate': float(df['process_cores_estimate'].max()),
    'mean_process_cores_estimate': float(df['process_cores_estimate'].mean()),
    'max_active_threads_estimate': int(df['active_threads_estimate'].max()),
}
display(pd.DataFrame([summary]))

print('--- Script output (tail) ---')
for line in stdout_text.strip().splitlines()[-10:]:
    print(line)


In [ ]:
per_core = np.asarray(per_core_samples, dtype=float)
if per_core.ndim != 2:
    raise RuntimeError('Per-core samples are malformed.')

t = df['time_s'].to_numpy()
fig, axes = plt.subplots(4, 1, figsize=(12, 14), gridspec_kw={'height_ratios': [2.2, 1.2, 1.2, 1.3]})

im = axes[0].imshow(
    per_core.T,
    aspect='auto',
    origin='lower',
    cmap='magma',
    vmin=0,
    vmax=100,
    extent=[float(t.min()), float(t.max()), 0, per_core.shape[1] - 1],
)
axes[0].set_ylabel('CPU core index')
axes[0].set_title('System per-core utilization heatmap during CPU bitset run')
cbar = fig.colorbar(im, ax=axes[0], pad=0.01)
cbar.set_label('CPU utilization (%)')

axes[1].plot(t, df['system_active_cores'], color='tab:blue', linewidth=1.8, label=f'System active cores (>= {active_core_threshold_pct:.1f}%)')
axes[1].plot(t, df['process_cores_estimate'], color='tab:orange', linewidth=1.8, label='Process cores estimate')
axes[1].set_ylabel('Core count')
axes[1].grid(alpha=0.25)
axes[1].legend(loc='upper right')

axes[2].plot(t, df['process_cpu_percent'], color='tab:red', linewidth=1.8, label='Process CPU (%)')
axes[2].plot(t, df['system_mean_cpu_percent'], color='tab:purple', linewidth=1.2, linestyle='--', label='System mean CPU (%)')
axes[2].set_ylabel('CPU (%)')
axes[2].grid(alpha=0.25)
axes[2].legend(loc='upper right')

axes[3].plot(t, df['active_threads_estimate'], color='tab:green', linewidth=1.8, label='Active process threads (estimated)')
axes[3].plot(t, df['num_threads'], color='tab:gray', linewidth=1.2, linestyle='--', label='Total process threads')
axes[3].set_xlabel('Time (s)')
axes[3].set_ylabel('Thread count')
axes[3].grid(alpha=0.25)
axes[3].legend(loc='upper right')

plt.tight_layout()
out_img = script_path.parent / 'cores_usage_dashboard_bitset.png'
plt.savefig(out_img, dpi=300, format='png')
print(f'Saved dashboard to: {out_img}')
plt.show()


In [ ]:
out_csv = script_path.parent / 'cpu_usage_samples_bitset.csv'
df.to_csv(out_csv, index=False)
print(f'Saved samples to: {out_csv}')
